<a href="https://colab.research.google.com/github/sathkumara-smna/258739K/blob/main/site_power_cell_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
# Cell-Level Hybrid Engineering + ML Telecom Power Prediction
## Google Colab Python Code
# ============================================================
# CELL-LEVEL HYBRID ENGINEERING + ML MODEL
# TELECOM SITE POWER PREDICTION
# ============================================================

# ============================================================
# IMPORT LIBRARIES
# ============================================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.ensemble import RandomForestRegressor
# ============================================================
# LOAD FILES FROM GITHUB
# ============================================================

site_db_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/Site%20Database%20from%20Sey.xlsx"
site_power_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/site_power%20from%20Sey.xlsx"
traffic_4g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/4G%20Traffic.xlsx"
traffic_5g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/5G%20Traffic.xlsx"

# ============================================================
# READ EXCEL FILES
# ============================================================

site_db = pd.read_excel(site_db_url)
site_power = pd.read_excel(site_power_url)
traffic_4g = pd.read_excel(traffic_4g_url)
traffic_5g = pd.read_excel(traffic_5g_url)
site_db.head(2)

,#,Site_ID,Site Name,2G RRUs,3G RRUs,4G RRUs,5G AAUs,2G Boards,3G Boards,4G Boards,5G Boards,BBU 5900,BBU 3900,BBU 3910
0,1,101,AIRPORT_MAHE,3,7,12,3,1,2,2,1,1,1,1
1,2,102,AIRPORT_PRASLIN,2,4,4,0,1,1,1,0,0,1,0


In [28]:
# ============================================================
# RENAME SITE DATABASE COLUMNS
# ============================================================

site_db.columns = [
    '#',
    'Site_ID',
    'Site_Name',
    'RRU_2G',
    'RRU_3G',
    'RRU_4G',
    'AAU_5G',
    'Col_H',
    'Col_I',
    'Boards_4G',
    'Boards_5G',
    'BBU5900',
    'BBU3900',
    'BBU3910'
]
site_db.head(2)

,#,Site_ID,Site_Name,RRU_2G,RRU_3G,RRU_4G,AAU_5G,Col_H,Col_I,Boards_4G,Boards_5G,BBU5900,BBU3900,BBU3910
0,1,101,AIRPORT_MAHE,3,7,12,3,1,2,2,1,1,1,1
1,2,102,AIRPORT_PRASLIN,2,4,4,0,1,1,1,0,0,1,0


In [29]:
# ============================================================
# MERGE SITE INFO TO 4G DATA
# ============================================================

lte_df = traffic_4g.merge(
    site_db,
    on='Site_ID',
    how='left'
)
lte_df.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,#,Site_Name,RRU_2G,RRU_3G,RRU_4G,AAU_5G,Col_H,Col_I,Boards_4G,Boards_5G,BBU5900,BBU3900,BBU3910
0,101,10111,11,1,2026-03-01,2026-03-01 00:00,11.90,1,AIRPORT_MAHE,3,7,12,3,1,2,2,1,1,1,1
1,101,10111,11,2,2026-03-01,2026-03-01 00:15,12.03,1,AIRPORT_MAHE,3,7,12,3,1,2,2,1,1,1,1


In [30]:
# ============================================================
# MERGE SITE INFO TO 5G DATA
# ============================================================

nr_df = traffic_5g.merge(
    site_db,
    on='Site_ID',
    how='left'
)
nr_df.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,#,Site_Name,RRU_2G,RRU_3G,RRU_4G,AAU_5G,Col_H,Col_I,Boards_4G,Boards_5G,BBU5900,BBU3900,BBU3910
0,101,1011,1,1,2026-03-01,2026-03-01 00:00,61.20,1,AIRPORT_MAHE,3,7,12,3,1,2,2,1,1,1,1
1,101,1011,1,2,2026-03-01,2026-03-01 00:15,67.31,1,AIRPORT_MAHE,3,7,12,3,1,2,2,1,1,1,1


In [33]:
# ============================================================
# FILL NULLS
# ============================================================

lte_df.fillna(0, inplace=True)
nr_df.fillna(0, inplace=True)
print("Done")

Done


In [34]:
# ============================================================
# ENGINEERING POWER ASSUMPTIONS
# ============================================================

RRU_2G_POWER = 150
RRU_3G_POWER = 200
RRU_4G_POWER = 180
AAU_5G_POWER = 500

BBU3900_POWER = 55
BBU3910_POWER = 65
BBU5900_POWER = 75

BOARD_4G_POWER = 42.5
BOARD_5G_POWER = 80
print("Done")

Done


In [36]:
# ============================================================
# LTE RRU COUNT PER SITE
# ============================================================

lte_rru_count = (

    lte_df.groupby('Site_ID')['Cell_ID']
    .nunique()
    .to_dict()

)

lte_df['lte_rru_count'] = (
    lte_df['Site_ID'].map(lte_rru_count)
)
lte_df.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,#,Site_Name,RRU_2G,...,RRU_4G,AAU_5G,Col_H,Col_I,Boards_4G,Boards_5G,BBU5900,BBU3900,BBU3910,lte_rru_count
0,101,10111,11,1,2026-03-01,2026-03-01 00:00,11.90,1,AIRPORT_MAHE,3,...,12,3,1,2,2,1,1,1,1,12
1,101,10111,11,2,2026-03-01,2026-03-01 00:15,12.03,1,AIRPORT_MAHE,3,...,12,3,1,2,2,1,1,1,1,12


In [38]:
# ============================================================
# NR AAU COUNT PER SITE
# ============================================================
nr_aau_count = (

    nr_df.groupby('Site_ID')['Cell_ID']
    .nunique()
    .to_dict()

)

nr_df['nr_aau_count'] = (
    nr_df['Site_ID'].map(nr_aau_count)
)


In [39]:
# ============================================================
# LTE TRAFFIC BANDS
# ============================================================

lte_df['lte_traffic_max'] = np.select(

    [lte_df['traffic_load_mbps'] < 15,
     lte_df['traffic_load_mbps'] < 30,
     lte_df['traffic_load_mbps'] < 45,
     lte_df['traffic_load_mbps'] < 60,
     lte_df['traffic_load_mbps'] < 75,
     lte_df['traffic_load_mbps'] < 100],

    [15, 30, 45, 60, 75, 100],

    default=125
)
lte_df.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,#,Site_Name,RRU_2G,...,AAU_5G,Col_H,Col_I,Boards_4G,Boards_5G,BBU5900,BBU3900,BBU3910,lte_rru_count,lte_traffic_max
0,101,10111,11,1,2026-03-01,2026-03-01 00:00,11.90,1,AIRPORT_MAHE,3,...,3,1,2,2,1,1,1,1,12,15
1,101,10111,11,2,2026-03-01,2026-03-01 00:15,12.03,1,AIRPORT_MAHE,3,...,3,1,2,2,1,1,1,1,12,15


In [40]:
# ============================================================
# LTE POWER BANDS
# ============================================================

lte_df['bbu_power_band'] = np.select(

    [lte_df['traffic_load_mbps'] < 15,
     lte_df['traffic_load_mbps'] < 30,
     lte_df['traffic_load_mbps'] < 45,
     lte_df['traffic_load_mbps'] < 60,
     lte_df['traffic_load_mbps'] < 75,
     lte_df['traffic_load_mbps'] < 100],

    [2, 4, 6, 8, 10, 12],

    default=14
)

lte_df['board_power_band'] = np.select(

    [lte_df['traffic_load_mbps'] < 15,
     lte_df['traffic_load_mbps'] < 30,
     lte_df['traffic_load_mbps'] < 45,
     lte_df['traffic_load_mbps'] < 60,
     lte_df['traffic_load_mbps'] < 75,
     lte_df['traffic_load_mbps'] < 100],

    [3, 6, 9, 12, 15, 18],

    default=21
)

lte_df['rru_power_band'] = np.select(

    [lte_df['traffic_load_mbps'] < 15,
     lte_df['traffic_load_mbps'] < 30,
     lte_df['traffic_load_mbps'] < 45,
     lte_df['traffic_load_mbps'] < 60,
     lte_df['traffic_load_mbps'] < 75,
     lte_df['traffic_load_mbps'] < 100],

    [10, 20, 30, 40, 50, 55],

    default=60
)
lte_df.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,#,Site_Name,RRU_2G,...,Boards_4G,Boards_5G,BBU5900,BBU3900,BBU3910,lte_rru_count,lte_traffic_max,bbu_power_band,board_power_band,rru_power_band
0,101,10111,11,1,2026-03-01,2026-03-01 00:00,11.90,1,AIRPORT_MAHE,3,...,2,1,1,1,1,12,15,2,3,10
1,101,10111,11,2,2026-03-01,2026-03-01 00:15,12.03,1,AIRPORT_MAHE,3,...,2,1,1,1,1,12,15,2,3,10


In [41]:
# ============================================================
# LTE ENGINEERING POWER
# ============================================================

lte_df['bbu3900_bp'] = np.ceil(((lte_df['BBU3900'] * BBU3900_POWER) / lte_df['lte_rru_count']) * 100) / 100
lte_df['bbu3910_bp'] = np.ceil(((lte_df['BBU3910'] * BBU3910_POWER) / lte_df['lte_rru_count']) * 100) / 100
lte_df['board_4g_bp'] = np.ceil(((lte_df['Boards_4G'] * BOARD_4G_POWER) / lte_df['lte_rru_count']) * 100) / 100
lte_df['rru_4g_bp'] = RRU_4G_POWER
lte_df['bbu_extra_power'] = np.ceil(((((lte_df['traffic_load_mbps'] / lte_df['lte_traffic_max']) * lte_df['bbu_power_band']) / lte_df['lte_rru_count']) * 100)) / 100
lte_df['board_extra_power'] = np.ceil(((((lte_df['traffic_load_mbps'] / lte_df['lte_traffic_max']) * lte_df['board_power_band']) / lte_df['lte_rru_count']) * 100)) / 100
lte_df['rru_extra_power'] = np.ceil((((lte_df['traffic_load_mbps'] / lte_df['lte_traffic_max']) * lte_df['rru_power_band']) * 100)) / 100
lte_df['calc_lte_sec_power'] = (

    lte_df['bbu3900_bp'] +
    lte_df['bbu3910_bp'] +
    lte_df['board_4g_bp'] +
    lte_df['rru_4g_bp'] +
    lte_df['bbu_extra_power'] +
    lte_df['board_extra_power'] +
    lte_df['rru_extra_power']

)
lte_df.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,#,Site_Name,RRU_2G,...,board_power_band,rru_power_band,bbu3900_bp,bbu3910_bp,board_4g_bp,rru_4g_bp,bbu_extra_power,board_extra_power,rru_extra_power,calc_lte_sec_power
0,101,10111,11,1,2026-03-01,2026-03-01 00:00,11.90,1,AIRPORT_MAHE,3,...,3,10,4.59,5.42,7.09,180,0.14,0.20,7.94,205.38
1,101,10111,11,2,2026-03-01,2026-03-01 00:15,12.03,1,AIRPORT_MAHE,3,...,3,10,4.59,5.42,7.09,180,0.14,0.21,8.02,205.47


In [42]:
# ============================================================
# NR TRAFFIC BANDS
# ============================================================

nr_df['max_5g_traffic'] = np.select(

    [nr_df['traffic_load_mbps'] < 60,
     nr_df['traffic_load_mbps'] < 200,
     nr_df['traffic_load_mbps'] < 400,
     nr_df['traffic_load_mbps'] < 800,
     nr_df['traffic_load_mbps'] < 1200,
     nr_df['traffic_load_mbps'] < 1300],

    [60, 200, 400, 800, 1200, 1300],

    default=1300
)

In [43]:
# ============================================================
# NR POWER BANDS
# ============================================================

nr_df['bbu_power_band'] = np.select(

    [nr_df['traffic_load_mbps'] < 60,
     nr_df['traffic_load_mbps'] < 200,
     nr_df['traffic_load_mbps'] < 400,
     nr_df['traffic_load_mbps'] < 800,
     nr_df['traffic_load_mbps'] < 1200,
     nr_df['traffic_load_mbps'] < 1300],

    [2, 4, 6, 8, 10, 12],

    default=14
)

nr_df['board_power_band'] = np.select(

    [nr_df['traffic_load_mbps'] < 60,
     nr_df['traffic_load_mbps'] < 200,
     nr_df['traffic_load_mbps'] < 400,
     nr_df['traffic_load_mbps'] < 800,
     nr_df['traffic_load_mbps'] < 1200,
     nr_df['traffic_load_mbps'] < 1300],

    [5, 10, 15, 20, 25, 30],

    default=35
)

nr_df['aau_power_band'] = np.select(

    [nr_df['traffic_load_mbps'] < 60,
     nr_df['traffic_load_mbps'] < 200,
     nr_df['traffic_load_mbps'] < 400,
     nr_df['traffic_load_mbps'] < 800,
     nr_df['traffic_load_mbps'] < 1200,
     nr_df['traffic_load_mbps'] < 1300],

    [50, 100, 150, 200, 250, 300],

    default=350
)

In [44]:
# ============================================================
# NR ENGINEERING POWER
# ============================================================

nr_df['bbu5900_bp'] = np.ceil(((nr_df['BBU5900'] * BBU5900_POWER) / nr_df['nr_aau_count']) * 100) / 100
nr_df['board_5g_bp'] = np.ceil(((nr_df['Boards_5G'] * BOARD_5G_POWER) / nr_df['nr_aau_count']) * 100) / 100
nr_df['aau_5g_bp'] = AAU_5G_POWER
nr_df['bbu_extra_power'] = np.ceil(((((nr_df['traffic_load_mbps'] / nr_df['max_5g_traffic']) * nr_df['bbu_power_band']) / nr_df['nr_aau_count']) * 100)) / 100
nr_df['board_extra_power'] = np.ceil(((((nr_df['traffic_load_mbps'] / nr_df['max_5g_traffic']) * nr_df['board_power_band']) / nr_df['nr_aau_count']) * 100)) / 100
nr_df['aau_extra_power'] = np.ceil((((nr_df['traffic_load_mbps'] / nr_df['max_5g_traffic']) * nr_df['aau_power_band']) * 100)) / 100
nr_df['calc_5g_sec_power'] = (

    nr_df['bbu5900_bp'] +
    nr_df['board_5g_bp'] +
    nr_df['aau_5g_bp'] +
    nr_df['bbu_extra_power'] +
    nr_df['board_extra_power'] +
    nr_df['aau_extra_power']

)
nr_df.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,#,Site_Name,RRU_2G,...,bbu_power_band,board_power_band,aau_power_band,bbu5900_bp,board_5g_bp,aau_5g_bp,bbu_extra_power,board_extra_power,aau_extra_power,calc_5g_sec_power
0,101,1011,1,1,2026-03-01,2026-03-01 00:00,61.20,1,AIRPORT_MAHE,3,...,4,10,100,25.0,26.67,500,0.41,1.02,30.60,583.70
1,101,1011,1,2,2026-03-01,2026-03-01 00:15,67.31,1,AIRPORT_MAHE,3,...,4,10,100,25.0,26.67,500,0.45,1.13,33.66,586.91


In [45]:
# ============================================================
# LTE SITE POWER AGGREGATION
# ============================================================

lte_site_power = (

    lte_df.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['calc_lte_sec_power']

    .sum()

)
lte_site_power.head(2)

,Site_ID,trigger_ID,date,datetime,calc_lte_sec_power
0,101,1,2026-03-01,2026-03-01 00:00,2470.12
1,101,1,2026-03-02,2026-03-02 00:00,2472.43


In [57]:
# ============================================================
# NR SITE POWER AGGREGATION
# ============================================================

nr_site_power = (

    nr_df.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['calc_5g_sec_power']

    .sum()

)
nr_site_power.head(2)

,Site_ID,trigger_ID,date,datetime,calc_5g_sec_power
0,101,1,2026-03-01,2026-03-01 00:00,1752.97
1,101,1,2026-03-02,2026-03-02 00:00,1777.56


In [58]:
# ============================================================
# MERGE LTE + NR SITE POWER
# ============================================================

site_prediction = lte_site_power.merge(

    nr_site_power,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='outer'

)

site_prediction.fillna(0, inplace=True)
site_prediction.head(2)

,Site_ID,trigger_ID,date,datetime,calc_lte_sec_power,calc_5g_sec_power
0,101,1,2026-03-01,2026-03-01 00:00,2470.12,1752.97
1,101,1,2026-03-02,2026-03-02 00:00,2472.43,1777.56


In [59]:
# ============================================================
# MERGE ACTUAL SITE POWER
# ============================================================

site_prediction = site_prediction.merge(

    site_power,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

site_prediction.fillna(0, inplace=True)
site_prediction.head(2)

,Site_ID,trigger_ID,date,datetime,calc_lte_sec_power,calc_5g_sec_power,site_power
0,101,1,2026-03-01,2026-03-01 00:00,2470.12,1752.97,6517.6299
1,101,1,2026-03-02,2026-03-02 00:00,2472.43,1777.56,6467.9894


In [60]:
# ============================================================
# ADD 2G + 3G POWER
# ============================================================

site_prediction = site_prediction.merge(

    site_db[['Site_ID', 'RRU_2G', 'RRU_3G']],
    on='Site_ID',
    how='left'

)

site_prediction['power_2g'] = (
    site_prediction['RRU_2G'] * RRU_2G_POWER
)

site_prediction['power_3g'] = (
    site_prediction['RRU_3G'] * RRU_3G_POWER
)
site_prediction.head(2)

,Site_ID,trigger_ID,date,datetime,calc_lte_sec_power,calc_5g_sec_power,site_power,RRU_2G,RRU_3G,power_2g,power_3g
0,101,1,2026-03-01,2026-03-01 00:00,2470.12,1752.97,6517.6299,3,7,450,1400
1,101,1,2026-03-02,2026-03-02 00:00,2472.43,1777.56,6467.9894,3,7,450,1400


In [61]:
# ============================================================
# FINAL ENGINEERING PREDICTION
# ============================================================

site_prediction['engineering_predicted_power'] = (

    site_prediction['power_2g'] +
    site_prediction['power_3g'] +
    site_prediction['calc_lte_sec_power'] +
    site_prediction['calc_5g_sec_power']

)
site_prediction.head(2)

,Site_ID,trigger_ID,date,datetime,calc_lte_sec_power,calc_5g_sec_power,site_power,RRU_2G,RRU_3G,power_2g,power_3g,engineering_predicted_power
0,101,1,2026-03-01,2026-03-01 00:00,2470.12,1752.97,6517.6299,3,7,450,1400,6073.09
1,101,1,2026-03-02,2026-03-02 00:00,2472.43,1777.56,6467.9894,3,7,450,1400,6099.99


In [62]:
# ============================================================
# RESIDUAL ERROR
# ============================================================

site_prediction['residual_error'] = (

    site_prediction['site_power'] -
    site_prediction['engineering_predicted_power']

)
site_prediction.head(2)

,Site_ID,trigger_ID,date,datetime,calc_lte_sec_power,calc_5g_sec_power,site_power,RRU_2G,RRU_3G,power_2g,power_3g,engineering_predicted_power,residual_error
0,101,1,2026-03-01,2026-03-01 00:00,2470.12,1752.97,6517.6299,3,7,450,1400,6073.09,444.5399
1,101,1,2026-03-02,2026-03-02 00:00,2472.43,1777.56,6467.9894,3,7,450,1400,6099.99,367.9994


In [63]:
# ============================================================
# MACHINE LEARNING FEATURES
# ============================================================

features = [

    'power_2g',
    'power_3g',
    'calc_lte_sec_power',
    'calc_5g_sec_power',
    'engineering_predicted_power'

]

X = site_prediction[features]
y = site_prediction['residual_error']

In [64]:
# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42
)
print("Done")

In [ ]:
# ============================================================
# RANDOM FOREST MODEL
# ============================================================

rf_model = RandomForestRegressor(

    n_estimators=100,
    random_state=42

)

rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)
print("Done")

In [ ]:
# ============================================================
# FINAL PREDICTIONS
# ============================================================

engineering_test = site_prediction.loc[
    X_test.index,
    'engineering_predicted_power'
]

final_predictions = (
    engineering_test + rf_predictions
)

actual_values = site_prediction.loc[
    X_test.index,
    'site_power'
]

In [ ]:
# ============================================================
# MODEL EVALUATION
# ============================================================

mae = mean_absolute_error(
    actual_values,
    final_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        actual_values,
        final_predictions
    )
)

mape = np.mean(

    np.abs(
        (actual_values - final_predictions)
        /
        actual_values
    )

) * 100

r2 = r2_score(
    actual_values,
    final_predictions
)

In [ ]:
# ============================================================
# PRINT RESULTS
# ============================================================

print('================================')
print('MODEL PERFORMANCE')
print('================================')

print(f'MAE  : {round(mae, 2)}')
print(f'RMSE : {round(rmse, 2)}')
print(f'MAPE : {round(mape, 2)} %')
print(f'R2   : {round(r2, 4)}')

In [ ]:
# ============================================================
# FINAL OUTPUT TABLE
# ============================================================

results_df = site_prediction.loc[

    X_test.index,
    [
        'Site_ID',
        'trigger_ID',
        'date',
        'datetime',
        'site_power',
        'engineering_predicted_power'
    ]

].copy()

results_df['ml_correction'] = rf_predictions

results_df['final_predicted_power'] = final_predictions

results_df['error'] = (

    results_df['site_power'] -
    results_df['final_predicted_power']

)

results_df['error_percentage'] = (

    np.abs(results_df['error'])
    /
    results_df['site_power']

) * 100

In [ ]:
# ============================================================
# EXPORT RESULTS
# ============================================================

results_df.to_excel(
    'Final_Cell_Level_Hybrid_Predictions.xlsx',
    index=False
)

print('================================')
print('OUTPUT FILE CREATED')
print('================================')

print('Final_Cell_Level_Hybrid_Predictions.xlsx')

# ============================================================
# SAMPLE RESULTS
# ============================================================

print(results_df.head(20))
